# Rank-deadline coverage: reproducible cross-panel study

This notebook reproduces the development evaluation and the frozen external PKIS1 validation for **rank-deadline profile coverage**. It uses public kinase-profile data, a standard Colab CPU, no model training, and no paid API.

**Scientific status.** The external mean-gain composite failed. The algorithm's deterministic first-hit delay bound held. The notebook reports both outcomes and does not retune the method. These retrospective biochemical labels do not establish cellular engagement, toxicity, efficacy, clinical safety, or therapeutic suitability.

In [ ]:
from pathlib import Path
import hashlib, json, os, shutil, subprocess, sys, urllib.request

REPO_URL = 'https://github.com/grewalsk/compiled-kinase-counterscreening.git'
REPO = Path('/content/compiled-kinase-counterscreening')
RESULTS = Path('/content/rank_deadline_results')
INPUTS = Path('/content/rank_deadline_inputs')
RUN_FULL_INFERENCE = True       # False is a code-path smoke test, not a paper rerun
RUN_EXPLORATORY_ROBUSTNESS = True
WORKERS = max(1, min(2, os.cpu_count() or 1))
PKIS2_URL = 'https://doi.org/10.1371/journal.pone.0181585.s004'
PKIS2_SHA256 = '48ead22a1f860cd0d5096fa87d5acd329f722fe8d65e693bb0be682a333e2a2c'
PKIS1_URL = ('https://raw.githubusercontent.com/SpencerEricksen/informers/'
             '5fd3934f5789c371026fc9eece1846ff1294122b/'
             'data/original_data/pkis1/PKIS_screening_data.csv.gz')
PKIS1_SHA256 = '81d7f9f82f7ee8e6b0f38dafe523da7254aaabe9449758a056372511d7868ad0'
print({'workers': WORKERS, 'full_inference': RUN_FULL_INFERENCE,
       'exploratory_robustness': RUN_EXPLORATORY_ROBUSTNESS})

## 1. Fetch the audited code and install pinned CPU dependencies

In [ ]:
if REPO.exists():
    subprocess.run(['git', '-C', str(REPO), 'pull', '--ff-only'], check=True)
else:
    subprocess.run(['git', 'clone', REPO_URL, str(REPO)], check=True)
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q',
                '-r', str(REPO / 'requirements_method_colab.txt')], check=True)
print(subprocess.run(['git', '-C', str(REPO), 'rev-parse', 'HEAD'],
                     check=True, text=True, capture_output=True).stdout.strip())

## 2. Download and byte-verify the two public PKIS sources

PKIS2 is the CC BY 4.0 PLOS supplement. PKIS1 is a ChEMBL-derived archived export and remains subject to ChEMBL CC BY-SA 3.0 attribution/share-alike terms. The Klaeger/ChEMBL-derived compact records are already versioned in the repository under the same ChEMBL terms.

In [ ]:
INPUTS.mkdir(parents=True, exist_ok=True)
PKIS2 = INPUTS / 'pkis2_s4.xlsx'
PKIS1 = INPUTS / 'PKIS_screening_data.csv.gz'

def digest(path):
    h = hashlib.sha256()
    with open(path, 'rb') as handle:
        for block in iter(lambda: handle.read(1024 * 1024), b''):
            h.update(block)
    return h.hexdigest()

def fetch_verified(url, path, expected):
    if not path.exists() or digest(path) != expected:
        urllib.request.urlretrieve(url, path)
    observed = digest(path)
    print(path.name, observed)
    assert observed == expected, f'Unexpected source bytes for {path.name}'

fetch_verified(PKIS2_URL, PKIS2, PKIS2_SHA256)
fetch_verified(PKIS1_URL, PKIS1, PKIS1_SHA256)

## 3. Run deterministic tests and validate the external input construction

In [ ]:
ENV = os.environ.copy()
ENV['PYTHONPATH'] = str(REPO / 'src')
ENV['MPLCONFIGDIR'] = '/content/matplotlib-cache'

def run(args):
    print('\n$', ' '.join(map(str, args)), flush=True)
    subprocess.run(list(map(str, args)), cwd=REPO, env=ENV, check=True)

tests = [
    'src/test_compiled_coverage.py',
    'src/test_conformal_coverage.py',
    'src/test_risk_controlled_coverage.py',
    'src/test_pkis1_external_validation.py',
]
run([sys.executable, '-m', 'unittest', '-v', *tests])
run([sys.executable, 'src/run_pkis1_external_validation.py',
     '--pkis1-raw', PKIS1, '--pkis2-xlsx', PKIS2,
     '--data-dir', REPO / 'data/derived',
     '--output-dir', RESULTS / 'input_validation',
     '--validate-input-only'])

## 4. Reproduce the development sequence

The runner contains the prespecified marginal, unconstrained-coverage, bounded-coverage, and failed conformal wrapper conditions. The paper's central comparison is fixed bounded coverage (`K=9`) versus chemical-similarity-weighted marginal ranking.

In [ ]:
if RESULTS.exists():
    shutil.rmtree(RESULTS)
RESULTS.mkdir(parents=True)
DEV = RESULTS / 'development'
cmd = [sys.executable, 'src/run_risk_controlled_coverage.py',
       '--data-dir', REPO / 'data/derived', '--pkis2-xlsx', PKIS2,
       '--output-dir', DEV, '--workers', WORKERS]
if not RUN_FULL_INFERENCE:
    cmd.append('--smoke-test')
run(cmd)
run([sys.executable, 'src/audit_bounded_component.py',
     '--summary', DEV / 'summary.json',
     '--stability', DEV / 'stability_diagnostics.csv.gz',
     '--output', DEV / 'bounded_component_audit.json'])
run([sys.executable, 'src/audit_bounded_detailed_metrics.py',
     '--case-metrics', DEV / 'case_metrics.csv.gz',
     '--output-json', DEV / 'bounded_detailed_metrics.json',
     '--output-csv', DEV / 'bounded_detailed_metrics.csv'])

## 5. Reproduce the frozen external PKIS1 test

In [ ]:
EXT = RESULTS / 'external_pkis1'
cmd = [sys.executable, 'src/run_pkis1_external_validation.py',
       '--pkis1-raw', PKIS1, '--pkis2-xlsx', PKIS2,
       '--data-dir', REPO / 'data/derived', '--output-dir', EXT,
       '--workers', WORKERS]
if not RUN_FULL_INFERENCE:
    cmd.append('--smoke-test')
run(cmd)

## 6. Optional post-result threshold and target-resolution audit

This 18-cell analysis was frozen only after the external failure. It tests whether that failure is a threshold or parent-target-collapse artifact; it cannot rescue or replace the frozen result.

In [ ]:
if RUN_EXPLORATORY_ROBUSTNESS and RUN_FULL_INFERENCE:
    run([sys.executable, 'src/audit_pkis1_thresholds.py',
         '--pkis1-raw', PKIS1, '--pkis2-xlsx', PKIS2,
         '--data-dir', REPO / 'data/derived',
         '--output-dir', RESULTS / 'exploratory_pkis1_robustness',
         '--workers', WORKERS])
else:
    print('Exploratory robustness skipped.')

## 7. Read the result without outcome switching

In [ ]:
import pandas as pd
from IPython.display import Image, display

dev = json.loads((DEV / 'summary.json').read_text())
ext = json.loads((EXT / 'summary.json').read_text())
print('Development status:', dev['status'])
print('External status:', ext['status'])
print('External composite passed:',
      ext['frozen_external_success_criteria']['composite_success'])
rows = []
for condition, comparisons in ext['comparisons'].items():
    effect = comparisons['bounded_minus_marginal']
    rows.append({
        'condition': condition,
        'bounded_minus_marginal_AUDC': effect['estimate'],
        'ci_low': effect['ci95'][0],
        'ci_high': effect['ci95'][1],
        'large_delays_ge_10': effect['large_delay_ge_10'],
    })
display(pd.DataFrame(rows))
display(Image(filename=str(EXT / 'figures/pkis1_external_effects.png')))
print('Interpretation: the universal mean-gain claim failed externally; '
      'the predeclared displacement/first-hit backstop held.')

## 8. Export every rerun artifact

In [ ]:
for name in [
    'RISK_CONTROLLED_COVERAGE_PROTOCOL.md',
    'PKIS1_EXTERNAL_VALIDATION_PROTOCOL.md',
    'PKIS1_EXTERNAL_VALIDATION_AMENDMENT.md',
    'PKIS1_EXTERNAL_VALIDATION_AMENDMENT_2.md',
    'PKIS1_ROBUSTNESS_AUDIT_PROTOCOL.md',
    'PKIS1_EXTERNAL_VALIDATION_RESULTS.md',
    'BOUNDED_COVERAGE_EXPERT_REVIEW.md',
]:
    shutil.copy2(REPO / name, RESULTS / name)
archive = shutil.make_archive('/content/rank_deadline_results', 'zip',
                              root_dir=RESULTS)
print('Created', archive, 'SHA-256', digest(Path(archive)))
try:
    from google.colab import files
    files.download(archive)
except ImportError:
    print('Outside Colab; archive left at', archive)